# NULLXES GAG CAP VC — Colab CPU deployment

Run this notebook only in Google Colab. Every network action is an explicit cell. It does not use local project execution, fabricated market observations, a public tunnel, or GPU inference. Drive mount prompts for your permission. Keep provider keys in Colab Secrets. The notebook is preparation only until you run it and retain the actual output.

## 1. Verify CPU runtime

In [ ]:
import os, platform
assert 'COLAB_RELEASE_TAG' in os.environ, 'Open this notebook in Google Colab first'
assert not os.path.exists('/dev/nvidia0'), 'Select a CPU runtime in Runtime > Change runtime type'
print('Python:', platform.python_version(), '| CPU cores:', os.cpu_count(), '| GPU: disabled')

## 2. Mount Drive after interactive consent

In [ ]:
from google.colab import drive
drive.mount('/content/drive')  # Google presents its normal permission prompt
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/NULLXES/GAG-CAP-VC')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

## 3. Retrieve the project from Drive

Copy the reviewed project folder to `MyDrive/NULLXES/GAG-CAP-VC/project` before running this cell. This avoids assuming a public Git repository exists.

In [ ]:
PROJECT = DRIVE_ROOT / 'project'
assert (PROJECT / 'pyproject.toml').is_file(), f'Project files not found: {PROJECT}'
os.chdir(PROJECT)
print('Project source:', PROJECT)

## 4. Install dependencies inside Colab

In [ ]:
%pip install '.[terminal,calibration,dev]'
# llama-cpp-python CPU build is isolated in the later Qwen section. No CUDA wheel is installed.

## 5. Configure keys and check official API reachability

A key is not printed. Network requests are made only when this cell is run. API availability is not proof that historical coverage or permissions are sufficient.

In [ ]:
import requests
from google.colab import userdata
from IPython.display import display
checks = {}
for name, url, params in [
    ('CBR', 'https://www.cbr.ru/DailyInfoWebServ/DailyInfo.asmx/KeyRateXML', {'from':'01/01/2025','To':'31/01/2025'}),
    ('MOEX', 'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/SBER/candles.json', {'from':'2025-01-01','till':'2025-01-31','interval':'24'})
]:
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        checks[name] = f'REACHABLE HTTP {response.status_code}'
    except requests.RequestException as exc:
        checks[name] = f'UNAVAILABLE {type(exc).__name__}: {exc}'
try:
    os.environ['FRED_API_KEY'] = userdata.get('FRED_API_KEY')
    response = requests.get('https://api.stlouisfed.org/fred/series/observations', params={'series_id':'CPIAUCSL','api_key':os.environ['FRED_API_KEY'],'file_type':'json','limit':'1'}, timeout=30)
    response.raise_for_status(); checks['FRED'] = f'REACHABLE HTTP {response.status_code}'
except Exception as exc:
    checks['FRED'] = f'UNAVAILABLE {type(exc).__name__} (detail redacted)'
display(checks)

## 6–7. Ingest and persist actual provider data

MOEX use may require a separate license. The chosen securities and board codes must be reviewed. This cell creates no substitute observations.

In [ ]:
from datetime import date
from gagcap.ingestion.cbr import key_rate
from gagcap.ingestion.moex import candles
from gagcap.ingestion.fred import observations as fred_observations
from gagcap.storage.lake import ObservationLake
lake = ObservationLake(DRIVE_ROOT / 'lake')
start, end = date(2015, 1, 1), date.today()
ingest_status, real_rows = {}, []
for name, fetch in [('CBR', lambda: key_rate(start, end)),
                    ('MOEX_SBER', lambda: candles('SBER', start, end, engine='stock', market='shares', board='TQBR', unit='RUB_per_share')),
                    ('FRED_CPIAUCSL', lambda: fred_observations('CPIAUCSL', start, end, vintage_as_of=end, unit='index_1982_84_100'))]:
    try:
        batch = fetch(); real_rows.extend(batch); lake_path = lake.write(batch)
        ingest_status[name] = f'INGESTED {len(batch)}'
    except Exception as exc:
        ingest_status[name] = f'DATA_UNAVAILABLE {type(exc).__name__}: {exc}'
print(ingest_status)
if not any(row.source == 'moex' for row in real_rows) or not any(row.source in {'cbr','fred'} for row in real_rows):
    raise RuntimeError('Acceptance prerequisites missing: require one real MOEX series and one real macro series')
print(f'Persisted {len(real_rows)} real observations to {lake_path}')

## 8. Run unit and integration test suite in Colab

In [ ]:
os.environ['GAGCAP_RUN_NETWORK_INTEGRATION'] = '1'
!python -m pytest -q

## 9. Multi-horizon walk-forward evaluation

This baseline cell reports observed results only. It uses daily market sessions and does not claim predictive skill. Each horizon is recorded in sessions; calendar horizon mapping is a documented approximation, not an exchange calendar.

In [ ]:
import pandas as pd
from gagcap.forecasting.backtesting import walk_forward, persist_experiment
from gagcap.forecasting.baselines import random_walk
from gagcap.forecasting.metrics import mae, rmse
market = lake.query("SELECT observation_date, value FROM observations WHERE series_id LIKE 'SBER:%' ORDER BY observation_date")
if market.empty: raise RuntimeError('DATA_UNAVAILABLE: no SBER observations')
series = pd.Series(market.value.to_numpy(), index=pd.DatetimeIndex(market.observation_date))
all_records = []
for calendar_days, steps in [(7,5),(30,21),(90,62)]:
    records = walk_forward(series, horizon=steps, min_train=252, predictor=lambda train,h: random_walk(train.iloc[-1], h), window='expanding')
    terminal_step = [record for record in records if record.horizon_steps == steps]
    if terminal_step:
        direction = sum((r.actual-r.origin_value == 0 and r.predicted-r.origin_value == 0) or ((r.actual-r.origin_value)*(r.predicted-r.origin_value) > 0) for r in terminal_step) / len(terminal_step)
        print(f'{calendar_days} calendar days (~{steps} sessions): n={len(terminal_step)}, MAE={mae([r.actual for r in terminal_step],[r.predicted for r in terminal_step]):.6f}, RMSE={rmse([r.actual for r in terminal_step],[r.predicted for r in terminal_step]):.6f}, directional_accuracy={direction:.4f}')
    else: print(f'{calendar_days} calendar days (~{steps} sessions): DATA_UNAVAILABLE (insufficient history)')
    all_records.extend(records)
if all_records: persist_experiment(all_records, str(DRIVE_ROOT / 'experiments' / 'baseline.parquet'))

## 10–11. Install CPU llama.cpp, download official Qwen GGUF in Colab, validate JSON

The model is downloaded only when this cell runs in Colab. Select a repository file explicitly from the official model repository. The download may consume significant Drive/runtime storage.

In [ ]:
%env CMAKE_ARGS=-DGGML_CUDA=OFF
%env FORCE_CMAKE=1
%pip install --no-binary=llama-cpp-python llama-cpp-python huggingface_hub
from huggingface_hub import HfApi, hf_hub_download
repo = 'Qwen/Qwen3-0.6B-GGUF'
files = [item for item in HfApi().list_repo_files(repo) if item.lower().endswith('.gguf')]
if not files: raise RuntimeError('QWEN_UNAVAILABLE: official repository currently exposes no GGUF files')
print('Official GGUF files:', files)
MODEL_FILE = next((item for item in files if 'q4_k_m' in item.lower()), files[0])
MODEL_PATH = hf_hub_download(repo_id=repo, filename=MODEL_FILE, local_dir=DRIVE_ROOT / 'models')
os.environ['QWEN_GGUF_PATH'] = MODEL_PATH
from gagcap.intelligence.qwen import extract_event
# Supply text copied from a real, reviewed source and its exact source URL. No sample article is fabricated here.
GEO_SOURCE_TEXT = ''
GEO_SOURCE_URL = ''
if not GEO_SOURCE_TEXT.strip() or not GEO_SOURCE_URL.startswith('https://'):
    print('QWEN_EXTRACTION_UNAVAILABLE: provide reviewed source text and exact HTTPS URL to proceed')
else:
    result = extract_event(GEO_SOURCE_TEXT, source_url=GEO_SOURCE_URL, model_path=MODEL_PATH)
    print(result.status, result.reason)
    if result.status != 'AVAILABLE': raise RuntimeError(f'Qwen JSON extraction unavailable or invalid: {result.status}')
    print(result.event.model_dump())

## 12. Produce a baseline forecast record

This cell demonstrates explicit persistence metadata. The naive random-walk value is a benchmark and must not be described as a predictive model result.

In [ ]:
from datetime import timedelta
import hashlib
from gagcap.quant.models import Forecast, calendar_horizon
from gagcap.storage.forecast_store import write_forecasts
origin = series.index[-1].date(); horizon = calendar_horizon(7)
dataset_hash = hashlib.sha256(Path(lake_path).read_bytes()).hexdigest()
forecast = Forecast('random_walk_baseline', origin, origin + timedelta(days=7), 7, horizon.sessions, float(series.iloc[-1]), None, None, None, 'MOEX provider close price', dataset_hash, 'last observation carried forward; no skill claim')
write_forecasts([forecast], DRIVE_ROOT / 'forecasts' / 'forecasts.parquet')
print(forecast.as_record())

## 13. Start the research terminal through Colab's authenticated port proxy

This serves only inside the Colab runtime and displays through Colab's own proxy. No ngrok, public URL, or local-machine server is created.

In [ ]:
import subprocess, sys, time
os.environ['GAGCAP_LAKE_PATH'] = str(lake_path)
terminal_log = open(DRIVE_ROOT / 'streamlit.log', 'a')
terminal = subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', str(PROJECT / 'app.py'), '--server.port', '8501', '--server.address', '0.0.0.0', '--server.headless', 'true'], cwd=PROJECT, stdout=terminal_log, stderr=terminal_log)
from google.colab import output
display(output.serve_kernel_port_as_iframe(8501, height=900))
print('Streamlit process:', terminal.pid, '| Colab authenticated iframe only')